# Advanced E-Commerce Data Science Analysis
## Online Retail Dataset 

**Dataset:** UCI Online Retail (UK-based, 2010–2011)  
**Objective:** Extract high-value, multi-dimensional business insights  

### Analytical Techniques Used:
1. Multi-dimensional analysis (3+ variables)
2. Cross-tabulation (Country × Segment)
3. Percentile analysis (Top 1%/5%/10%/20%)
4. Cohort analysis (monthly retention)
5. Outlier detection & classification
6. Ratio-based feature engineering (AOV, frequency, basket size)
7. Time-series pattern analysis (monthly/weekly/hourly)
8. Customer segmentation (VIP / High-Value / Regular / Low-Value)
9. Product performance ranking

In [ ]:
!pip install pandas numpy matplotlib seaborn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Plotting defaults for professional-looking charts
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 13
sns.set_style('whitegrid')
sns.set_palette('viridis')

print("✅ All libraries loaded successfully.")


# 1. Data Loading & Overview

We load the Online Retail dataset from the UCI Machine Learning Repository. This dataset contains **541,909 invoice-level transactions** from a UK-based online retailer selling unique all-occasion gifts.

### Columns:
| Column | Description |
|--------|-------------|
| InvoiceNo | Unique invoice number (prefix 'C' = cancelled) |
| StockCode | Unique product/item code |
| Description | Product name |
| Quantity | Number of units per transaction line |
| InvoiceDate | Date-time of the invoice |
| UnitPrice | Price per unit in GBP (£) |
| CustomerID | Unique customer identifier |
| Country | Customer's country |


In [ ]:
# Load the dataset from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
df_raw = pd.read_excel(url, engine='openpyxl')

# If you have the file locally, uncomment the line below instead:
# df_raw = pd.read_excel("Online Retail.xlsx", engine='openpyxl')

print(f"✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"   Columns: {list(df_raw.columns)}")
df_raw.head(10)

## Dataset Shape & Info

In [ ]:
print("=" * 70)
print("DATASET SHAPE & TYPES")
print("=" * 70)
print(f"Rows   : {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")
print()
print("Column Data Types:")
for col in df_raw.columns:
    print(f"  {col:15s} → {str(df_raw[col].dtype)}")
print()
df_raw.info()

## Missing Values & Data Quality

In [ ]:
print("=" * 70)
print("MISSING VALUES ANALYSIS")
print("=" * 70)

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct,
    'Dtype': df_raw.dtypes
})
print(missing_df)
print()

print("=" * 70)
print("DATA QUALITY CHECKS")
print("=" * 70)
print(f"Negative Quantities  : {(df_raw['Quantity'] < 0).sum():,} rows ({(df_raw['Quantity'] < 0).mean()*100:.2f}%)")
print(f"Zero/Negative Prices : {(df_raw['UnitPrice'] <= 0).sum():,} rows ({(df_raw['UnitPrice'] <= 0).mean()*100:.2f}%)")
print(f"Cancelled Invoices   : {df_raw['InvoiceNo'].astype(str).str.startswith('C').sum():,} rows")
print(f"Unique Customers     : {df_raw['CustomerID'].nunique():,}")
print(f"Unique Products      : {df_raw['StockCode'].nunique():,}")
print(f"Unique Countries     : {df_raw['Country'].nunique()}")
print(f"Date Range           : {df_raw['InvoiceDate'].min()} → {df_raw['InvoiceDate'].max()}")

## Detailed Statistical Summary

In [ ]:
print("=" * 70)
print("STATISTICAL SUMMARY (Beyond .describe())")
print("=" * 70)

for col in ['Quantity', 'UnitPrice']:
    series = df_raw[col]
    print(f"\n━━━ {col} ━━━")
    print(f"  Mean     : {series.mean():>12.2f}")
    print(f"  Median   : {series.median():>12.2f}")
    print(f"  Std Dev  : {series.std():>12.2f}")
    print(f"  Skewness : {series.skew():>12.2f}  {'(right-skewed)' if series.skew() > 1 else '(approx symmetric)' if abs(series.skew()) < 1 else '(left-skewed)'}")
    print(f"  Kurtosis : {series.kurtosis():>12.2f}  {'(heavy tails)' if series.kurtosis() > 3 else '(normal tails)'}")
    print(f"  IQR      : {series.quantile(0.75) - series.quantile(0.25):>12.2f}")
    print(f"  P5       : {series.quantile(0.05):>12.2f}")
    print(f"  P25      : {series.quantile(0.25):>12.2f}")
    print(f"  P75      : {series.quantile(0.75):>12.2f}")
    print(f"  P95      : {series.quantile(0.95):>12.2f}")
    print(f"  P99      : {series.quantile(0.99):>12.2f}")

print("\n📊 Note: High skewness and kurtosis indicate extreme outliers that need investigation.")


# 2. Data Cleaning

### Why we clean:
- **Missing CustomerIDs** → Can't track customer behavior without an ID
- **Cancelled orders** (InvoiceNo starts with 'C') → Didn't generate real revenue
- **Negative/zero quantities** → Returns or errors, not actual purchases
- **Zero/negative prices** → Free items or errors distort revenue calculations
- **Duplicates** → System errors that inflate counts

### Principle: Every row removed must be justified and logged.

In [ ]:
print("=" * 70)
print("DATA CLEANING PIPELINE")
print("=" * 70)

df = df_raw.copy()
cleaning_log = []
original_size = len(df)

# Step 1: Remove rows with missing CustomerID
before = len(df)
df = df.dropna(subset=['CustomerID'])
removed = before - len(df)
cleaning_log.append(('Missing CustomerID', removed))
print(f"✅ Step 1: Removed {removed:,} rows with missing CustomerID ({removed/original_size*100:.1f}%)")
print(f"   → WHY: Cannot perform customer-level analysis without customer identification")

# Step 2: Remove cancelled transactions
before = len(df)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
removed = before - len(df)
cleaning_log.append(('Cancelled Orders', removed))
print(f"\n✅ Step 2: Removed {removed:,} cancelled transactions ({removed/original_size*100:.1f}%)")
print(f"   → WHY: Cancellations did not generate actual revenue")

# Step 3: Remove non-positive quantities
before = len(df)
df = df[df['Quantity'] > 0]
removed = before - len(df)
cleaning_log.append(('Quantity ≤ 0', removed))
print(f"\n✅ Step 3: Removed {removed:,} rows with Quantity ≤ 0 ({removed/original_size*100:.1f}%)")
print(f"   → WHY: Negative = returns, zero = no actual purchase")

# Step 4: Remove non-positive prices
before = len(df)
df = df[df['UnitPrice'] > 0]
removed = before - len(df)
cleaning_log.append(('UnitPrice ≤ 0', removed))
print(f"\n✅ Step 4: Removed {removed:,} rows with UnitPrice ≤ 0 ({removed/original_size*100:.1f}%)")
print(f"   → WHY: Free items and negative prices distort revenue")

# Step 5: Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates()
removed = before - len(df)
cleaning_log.append(('Duplicates', removed))
print(f"\n✅ Step 5: Removed {removed:,} duplicate rows ({removed/original_size*100:.1f}%)")
print(f"   → WHY: System-generated duplicates inflate all metrics")

# Step 6: Fix data types
df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"\n{'='*70}")
print(f"📊 CLEANING COMPLETE")
print(f"   Original : {original_size:,} rows")
print(f"   Final    : {len(df):,} rows")
print(f"   Retained : {len(df)/original_size*100:.1f}%")
print(f"   Removed  : {original_size - len(df):,} rows ({(original_size-len(df))/original_size*100:.1f}%)")
print(f"{'='*70}")

## Cleaning Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart: Retained vs Removed
sizes = [original_size - len(df), len(df)]
labels = ['Removed', 'Retained']
colors = ['#FF6B6B', '#4ECDC4']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 14, 'fontweight': 'bold'},
            explode=(0.05, 0))
axes[0].set_title('Data Cleaning: Retained vs Removed', fontsize=16, fontweight='bold')

# Bar chart: Rows removed per step
step_labels = [c[0] for c in cleaning_log]
step_values = [c[1] for c in cleaning_log]
bars = axes[1].bar(step_labels, step_values,
                color=['#E74C3C', '#FF8C00', '#FFD700', '#9B59B6', '#3498DB'],
                edgecolor='black', linewidth=0.5)
axes[1].set_title('Rows Removed per Cleaning Step', fontsize=16, fontweight='bold')
axes[1].set_ylabel('Rows Removed')
axes[1].tick_params(axis='x', rotation=25)
for bar, val in zip(bars, step_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('cleaning_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Feature Engineering

### What is Feature Engineering?
Creating **new meaningful variables** from raw data. The raw dataset gives us `Quantity` and `UnitPrice`, but business decisions need metrics like **Revenue**, **AOV**, **Purchase Frequency**, and **Customer Lifespan**.

### Two levels of features:
1. **Transaction-level** — calculated per row (e.g., Revenue = Qty × Price)
2. **Customer-level** — aggregated per customer (e.g., Total Revenue, AOV, Frequency)

| Feature | Formula | Business Purpose |
|---------|---------|-----------------|
| Revenue | Quantity × UnitPrice | Core revenue metric per line item |
| AOV | TotalRevenue ÷ NumInvoices | Spending level per visit |
| PurchaseFrequency | NumInvoices ÷ (Lifespan/30) | Engagement intensity |
| AvgBasketSize | TotalQuantity ÷ NumInvoices | Volume per order |
| CustomerLifespan | LastPurchase − FirstPurchase | Loyalty indicator (in days) |
| RevenuePerProduct | TotalRevenue ÷ NumProducts | Product dependency risk |

In [ ]:
print("=" * 70)
print("FEATURE ENGINEERING — TRANSACTION LEVEL")
print("=" * 70)

# Revenue per line item
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Time-based features
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
df['InvoiceDay'] = df['InvoiceDate'].dt.date
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
df['Month'] = df['InvoiceDate'].dt.month
df['Year'] = df['InvoiceDate'].dt.year
df['WeekOfYear'] = df['InvoiceDate'].dt.isocalendar().week.astype(int)

print("✅ Created: Revenue, InvoiceMonth, InvoiceDay, DayOfWeek, Hour, Month, Year, WeekOfYear")
print(f"\n📊 Revenue column sample:")
print(f"   Min Revenue    : £{df['Revenue'].min():.2f}")
print(f"   Max Revenue    : £{df['Revenue'].max():,.2f}")
print(f"   Mean Revenue   : £{df['Revenue'].mean():.2f}")
print(f"   Total Revenue  : £{df['Revenue'].sum():,.2f}")
df[['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'Revenue', 'DayOfWeek', 'Hour']].head(10)


##  Customer-Level Features

In [ ]:
print("=" * 70)
print("FEATURE ENGINEERING — CUSTOMER LEVEL")
print("=" * 70)

# Aggregate all transactions per customer
customer_df = df.groupby('CustomerID').agg(
    TotalRevenue       = ('Revenue', 'sum'),
    TotalQuantity      = ('Quantity', 'sum'),
    NumInvoices        = ('InvoiceNo', 'nunique'),
    NumProducts        = ('StockCode', 'nunique'),
    AvgOrderValue      = ('Revenue', 'mean'),
    AvgUnitPrice       = ('UnitPrice', 'mean'),
    FirstPurchase      = ('InvoiceDate', 'min'),
    LastPurchase       = ('InvoiceDate', 'max'),
    NumCountries       = ('Country', 'nunique'),
    PrimaryCountry     = ('Country', lambda x: x.mode()[0])
).reset_index()

# Derived metrics
customer_df['AOV'] = customer_df['TotalRevenue'] / customer_df['NumInvoices']
customer_df['RevenuePerProduct'] = customer_df['TotalRevenue'] / customer_df['NumProducts']
customer_df['AvgBasketSize'] = customer_df['TotalQuantity'] / customer_df['NumInvoices']
customer_df['CustomerLifespan'] = (customer_df['LastPurchase'] - customer_df['FirstPurchase']).dt.days
customer_df['PurchaseFrequency'] = customer_df['NumInvoices'] / (customer_df['CustomerLifespan'].replace(0, 1) / 30)

print(f"✅ Customer-level dataset: {customer_df.shape[0]:,} customers × {customer_df.shape[1]} features")
print()

# Distribution of key metrics
print("━━━ KEY METRIC DISTRIBUTIONS ━━━")
for metric in ['TotalRevenue', 'AOV', 'NumInvoices', 'AvgBasketSize', 'PurchaseFrequency', 'CustomerLifespan']:
    s = customer_df[metric]
    print(f"\n  📊 {metric}:")
    print(f"     Mean   = {s.mean():>12,.2f}    Median = {s.median():>12,.2f}")
    print(f"     P10    = {s.quantile(0.10):>12,.2f}    P90    = {s.quantile(0.90):>12,.2f}")
    print(f"     P99    = {s.quantile(0.99):>12,.2f}    Max    = {s.max():>12,.2f}")

customer_df.head(10)